In [ ]:
import os
import sys

import collections
import xarray as xr
xr.set_options(keep_attrs=True)
import netCDF4 as nc
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
import metpy.calc as mp
from metpy.units import units
from scipy.stats import ttest_ind, ttest_rel
from datetime import datetime

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

# add path to custom functions
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path+"/py_functions")
from map_plot_tools import *
from colorbar_funcs import *
from data_funcs import *
from stats_funcs import *

In [ ]:
### +++ DATA PATHS +++ ###

runNames=['obs', 'E2pt1_PIctrl_restart', 'cam.1senv.cpld', 'cam.sam.1senv.cpld'] #, 'envelope1s_e2.1_cpld', 'maxtopo_E2pt1_PI_restart_new'] # E2.1 cpld simulations
keys=['obs', 'ctrl', 'ca1senv', 'ca+sa1senv'] #, '1senv-global', 'max-global'] # product key
diags=['u_1000','v_1000','sst','prec'] # diagnostics

# store file paths in dictionary
dpath0='/discover/nobackup/projects/giss/baldwin_nip/dmkumar' # top level data directory
files={}

for i,key in enumerate(keys):
    files[key] = {}
    
    if key in ['obs']:
        files[key]['u_1000'] = f'{dpath0}/obs_data/merra2/merra2.U.1980-2022.monthly.nc'
        files[key]['v_1000'] = f'{dpath0}/obs_data/merra2/merra2.V.1980-2022.monthly.nc'
        files[key]['sst'] = f'{dpath0}/obs_data/hadisst.gn.timeseries.1870-2020.nc'
        files[key]['prec'] = f'{dpath0}/obs_data/imerg.gn.timeseries.2001-2018.nc'
        
    elif key in ['ctrl', 'ca1senv', 'ca+sa1senv']: #, '1senv-global', 'max-global']:
        run=runNames[i]
        for varn in diags:
            files[key][varn] = f'{dpath0}/E2pt1/{run}/timeseries/{varn}_timeseries.{run}.nc'
            

In [ ]:
### +++ TOPOGRAPHY CONTOURS +++ ###

# e2.1 topo bc info
land_mask=xr.open_dataset(f'{dpath0}/topo_files/Z2HX2fromZ1QX1N.BS1.nc').fgrnd

default=xr.open_dataset(f'{dpath0}/topo_files/Z2HX2fromZ1QX1N.BS1.nc').zatmo
cs1env=xr.open_dataset(f'{dpath0}/topo_files/giss.1s-env.topo.cam.sam.nc').zatmo
c1env=xr.open_dataset(f'{dpath0}/topo_files/giss.1s-env.topo.cam_only.nc').zatmo  
maxtopo=xr.open_dataset(f'{dpath0}/topo_files/giss.maxtopo.nc').zatmo  

# generate topo masks
c1env_mask=c1env.where((c1env-default)>0, 0)
cs1env_mask=cs1env.where((cs1env-default)>0, 0)
maxtopo_mask=maxtopo.where((maxtopo-default)>0, 0)

In [ ]:
## ORGANIZE DATA AND CALCULATE CLIMATOLOGIES

keys=['obs', 'ctrl', 'ca1senv', 'ca+sa1senv']
diags=['u_1000','v_1000','sst','prec']

# number of years to omit 
omit_yrs = 30
n_omit = omit_yrs * 12
tmax = 350 * 12
latmin = -55
latmax = 30
lonmin = 90
lonmax = 355

# initialize dictionaries
dat = collections.defaultdict(dict)
time_mean = collections.defaultdict(dict)
seas_mean = collections.defaultdict(dict)
ann_seas_mean = collections.defaultdict(dict)

## Open and process timeseries files
print('Loading data for...')
for i,key in enumerate(keys):
    print(f'{key}')
    
    # initialize simulation sub-dicts
    dat[key] = {}
    
    for varn in diags:
        
        # process obs files
        if key in ['obs']:
            if varn in ['u_1000']:
                ds = xr.open_dataset(files[key][varn], chunks={}).U[:,0,:,:].rename({'lev':'plm'})
                ds_flip = longitude_flip(ds)
                ds_clip = ds_flip.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
                dat[key]['u_1000'] = ds_clip
                del ds
                del ds_flip
                del ds_clip
            if varn in ['v_1000']:
                ds = xr.open_dataset(files[key][varn], chunks={}).V[:,0,:,:].rename({'lev':'plm'})
                ds_flip = longitude_flip(ds)
                ds_clip = ds_flip.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
                dat[key]['v_1000'] = ds_clip
                del ds
                del ds_flip
                del ds_clip
            if varn in ['prec']:
                ds = xr.open_dataset(files[key][varn], chunks={}).precipitation * 24 # convert from mm/hr to mm/day
                ds = ds.transpose('time','lat','lon')
                ds.attrs['units'] = 'mm/day'
                ds.attrs['Units'] = 'mm/day'
                ds_flip = longitude_flip(ds)
                ds_clip = ds_flip.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
                dat[key]['prec'] = ds_clip
                del ds
                del ds_flip
                del ds_clip
            if varn in ['sst']:
                ds = xr.open_dataset(files[key][varn], chunks={}).sst.rename({'longitude':'lon', 'latitude':'lat'}).isel(lat=slice(None, None, -1))
                ds_flip = longitude_flip(ds)
                ds_clip = ds_flip.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
                dat[key]['sst'] = ds_clip
                del ds
                del ds_flip
                del ds_clip
    
        # process model simulation files
        elif key in ['ctrl', 'ca1senv', 'ca+sa1senv']: #, '1senv-global', 'max-global']:
            ds = xr.open_dataset(files[key][varn], chunks={})[varn].isel(time=slice(n_omit,tmax))
            ds_flip = longitude_flip(ds)
            ds_clip = ds_flip.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
            dat[key][varn] = ds_clip
            del ds
            del ds_flip
            del ds_clip
            
print('Done.')

## Calculate time-means
print('Calculating time means for...')
for i,key in enumerate(keys):
    print(f'{key}')
    
    # initialize simulation sub-dicts
    time_mean[key] = {}
    seas_mean[key] = {}
    ann_seas_mean[key] = {}
    
    # calculate near-surface wind speed
    for varn in ['spd_1000']:
        dat[key][varn] = windSpd(dat[key]['u_1000'],dat[key]['u_1000'])
    
    # calculate time-means
    for varn in ['u_1000','v_1000','spd_1000','sst','prec']:
        # calculate time-means
        time_mean[key][varn] = dat[key][varn].mean(dim='time')
        seas_mean[key][varn] = season_mean(dat[key][varn])
        ann_seas_mean[key][varn] = annual_season_mean(dat[key][varn])

print('Done.')

In [ ]:
## COMPARING ALL MODEL RUNS TO OBS.

# initialize dictionaries
obs_diff = collections.defaultdict(dict)
obs_diff_mask = collections.defaultdict(dict)
obs_ptvals = collections.defaultdict(dict)

seasons = ['DJF','MAM','JJA','SON']
diags=['u_1000','v_1000','spd_1000','sst','prec'] 


# need to re-grid obs data to E2.1 grid first
regrid= {}
seas_mean_regrid = {}
ann_seas_mean_regrid = collections.defaultdict(dict)
lats=time_mean['ctrl']['sst'].lat
lons=time_mean['ctrl']['sst'].lon

# need to re-grid obs data to E2.1 grid first
print('Interpolating obs data to model grid.')
for key in ['obs']:
    
    # initialize sub-dicts
    regrid[key] = {}
    seas_mean_regrid[key] = {}
    
    for varn in diags:
        # re-grid time mean obs
        regrid[key][varn] = time_mean[key][varn].interp(lat=lats, lon=lons, method='linear')
        # re-grid seasonal time-mean obs
        seas_mean_regrid[key][varn] = seas_mean[key][varn].interp(lat=lats, lon=lons, method='linear')
        # re-grid annual seasonal mean obs
        ann_seas_mean_regrid[key][varn] = {}
        for season in seasons:
            ann_seas_mean_regrid[key][varn][season] = ann_seas_mean[key][varn][season].interp(lat=lats, lon=lons, method='linear')
print('Done.')


print('Significance testing for...')
for key in ['ctrl', 'ca1senv', 'ca+sa1senv']:
    print(f'{key}')
    
    for varn in diags:
        obs_diff[key][varn] = {}
        obs_diff_mask[key][varn] = {}
        obs_ptvals[key][varn] = {}
        
        for season in seasons:
            # calculate significance of model - obs difference
            diff_, diff_mask_, ptvals_ = sigtest2n(ann_seas_mean_regrid['obs'][varn][season].values, ann_seas_mean[key][varn][season].values, 
                                                   seas_mean_regrid['obs'][varn].sel(season=season).values, seas_mean[key][varn].sel(season=season).values)
            obs_diff[key][varn][season] = diff_
            obs_diff_mask[key][varn][season] = diff_mask_
            obs_ptvals[key][varn][season] = ptvals_

print('Done.')

In [ ]:
cpld_usig = {}
cpld_vsig = {}

season = 'MAM'

for key in ['ctrl','ca1senv', 'ca+sa1senv']:
    cpld_usig[key] = np.where((obs_diff_mask[key]['u_1000'][season].mask==False) &
                              (obs_diff_mask[key]['v_1000'][season].mask==False),
                              obs_diff_mask[key]['u_1000'][season].data, np.nan)
    
    cpld_vsig[key] = np.where((obs_diff_mask[key]['u_1000'][season].mask==False) &
                              (obs_diff_mask[key]['v_1000'][season].mask==False),
                              obs_diff_mask[key]['v_1000'][season].data, np.nan)

## Calculate RMSE

In [ ]:
rmse=collections.defaultdict(dict)
diags=['spd_1000','sst','prec'] 
lat_min=-55
lat_max=25
lon_min=210 #90
lon_max=305 #355


print('Calculating RMSE between obs and...')
for key in ['ctrl','ca1senv', 'ca+sa1senv']:
    print(f'{key}')
    rmse[key] = {}
    
    for varn in diags:
        rmse[key][varn] = {}
        
        for season in ['DJF','MAM','JJA','SON']: 
            
            # Define actual and predicted values for domain of interest
            actual = seas_mean_regrid['obs'][varn].sel(season=season, lat=slice(lat_min,lat_max), lon=slice(lon_min,lon_max))
            
            predicted = seas_mean[key][varn].sel(season=season, lat=slice(lat_min,lat_max), lon=slice(lon_min,lon_max))

            # Calculate the mean squared error (MSE) by taking the mean of the squared differences
            meanSquaredError = ((predicted.load() - actual.load()) ** 2).mean()

            # Calculate the RMSE by taking the square root of the MSE
            rmse[key][varn][season] = np.sqrt(meanSquaredError)

            # Print the RMSE
            #print(f'The calculated RMSE of {varn} for {key} is: ' + str(rmse.values))
            
print('Done.')

In [ ]:
rmse_ocn=collections.defaultdict(dict)
diags=['spd_1000','sst','prec'] 
lat_min=-55
lat_max=25
lon_min=210
lon_max=305

# flip and clip land mask
land_mask_ = longitude_flip(land_mask)
land_mask__ = land_mask_.sel(lat=slice(lat_min,lat_max), lon=slice(lon_min,lon_max))

print('Calculating RMSE between obs and...')
for key in ['ctrl','ca1senv', 'ca+sa1senv']:
    print(f'{key}')
    rmse_ocn[key] = {}
    
    for varn in diags:
        rmse_ocn[key][varn] = {}
        
        for season in ['DJF','MAM','JJA','SON']: 
            
            # Define actual and predicted values for domain of interest
            actual = seas_mean_regrid['obs'][varn].sel(season=season, lat=slice(lat_min,lat_max), lon=slice(lon_min,lon_max)).where(land_mask__==0, np.nan)
            predicted = seas_mean[key][varn].sel(season=season, lat=slice(lat_min,lat_max), lon=slice(lon_min,lon_max)).where(land_mask__==0, np.nan)

            # Calculate the mean squared error (MSE) by taking the mean of the squared differences
            meanSquaredError = ((predicted.load() - actual.load()) ** 2).mean()

            # Calculate the RMSE by taking the square root of the MSE
            rmse_ocn[key][varn][season] = np.sqrt(meanSquaredError)

            # Print the RMSE
            #print(f'The calculated RMSE of {varn} for {key} is: ' + str(rmse.values))
            
print('Done.')

In [ ]:
rmse2=collections.defaultdict(dict)
diags=['spd_1000','sst','prec'] 
lat_min=-55
lat_max=25
lon_min=90
lon_max=355


print('Calculating RMSE between obs and...')
for key in ['ctrl','ca1senv', 'ca+sa1senv']:
    print(f'{key}')
    rmse2[key] = {}
    
    for varn in diags:
        rmse2[key][varn] = {}
        
        for season in ['DJF','MAM','JJA','SON']: 
            
            # Define actual and predicted values for domain of interest
            actual = seas_mean_regrid['obs'][varn].sel(season=season, lat=slice(lat_min,lat_max), lon=slice(lon_min,lon_max))
            predicted = seas_mean[key][varn].sel(season=season, lat=slice(lat_min,lat_max), lon=slice(lon_min,lon_max))

            # Calculate the mean squared error (MSE) by taking the mean of the squared differences
            meanSquaredError = ((predicted.load() - actual.load()) ** 2).mean()

            # Calculate the RMSE by taking the square root of the MSE
            rmse2[key][varn][season] = np.sqrt(meanSquaredError)

            # Print the RMSE
            #print(f'The calculated RMSE of {varn} for {key} is: ' + str(rmse.values))
            
print('Done.')

## Make Figure

In [ ]:
# -------------------- #
#       SETTINGS       #
# -------------------- #
# var info
season = 'MAM'
lons = time_mean['ctrl']['sst'].lon
lats = time_mean['ctrl']['sst'].lat
mlons = c1env_mask.lon
mlats = c1env_mask.lat
# colormap specs
wcmap=sns.diverging_palette(20, 220, as_cmap=True)
levels=np.linspace(-4.5, 4.5, 19)
wnorm=mpl.colors.BoundaryNorm(levels, wcmap.N)

scmap=cm.RdBu_r
levels=np.linspace(-3.5, 3.5, 15)
snorm=mpl.colors.BoundaryNorm(levels, scmap.N)

pcmap,_,_,_=get_settings(field='precip', diff=True)
levels=np.linspace(-6, 6, 25)
pnorm=mpl.colors.BoundaryNorm(levels, pcmap.N)
# vector specs
skip_n=2
w=0.0075
scalef=20
key_length=5
# text specs
text_kw={'weight':'bold', 'va':'center', 'size':'14', 'color':'black'}
text_kw2={'weight':'normal', 'va':'center', 'ha':'right', 'size':'12', 'color':'black'}
text_kw3={'weight':'normal', 'va':'center', 'ha':'right', 'size':'12', 'color':'blue'}
# map specs
proj=ccrs.PlateCarree(central_longitude=180)
trans=ccrs.PlateCarree()
bounds=[210,305,-55,25]
# label coordinates
tx=80
ty=35

# -------------------- #
#          FIG         #
# -------------------- #
fig,ax=plt.subplots(nrows=3, ncols=3, figsize=(14.5, 13), layout='constrained', subplot_kw={'projection': proj})


ax[0,0].text(tx,ty,'1000 hPa Winds', ha='center', **text_kw)
ax[0,1].text(tx,ty,'SST', ha='center', **text_kw)
ax[0,2].text(tx,ty,'Precipitation', ha='center', **text_kw)

ax[0,0].text(15,-15,'OBS $-$ CTRL', rotation=90, ha='center', **text_kw)
ax[1,0].text(15,-15,'OBS $-$ CA-ONLY', rotation=90, ha='center', **text_kw)
ax[2,0].text(15,-15,'OBS $-$ CA+SA', rotation=90, ha='center', **text_kw)


for i,key in enumerate(['ctrl', 'ca1senv', 'ca+sa1senv']):
    
    # winds
    cf1=ax[i,0].pcolormesh(lons, lats, obs_diff_mask[key]['spd_1000'][season], cmap=wcmap, norm=wnorm, transform=trans)
    q1=ax[i,0].quiver(lons[::skip_n], lats[::skip_n], cpld_usig[key][::skip_n,::skip_n], cpld_vsig[key][::skip_n,::skip_n],
                         color='k', width=w, scale=scalef, scale_units='inches', units='height', transform=trans, zorder=50)
    ax[i,0].text(120,27, ('RMSE = ' + "{:.2f}".format(rmse[key]['spd_1000'][season].values) + 'm/s'), **text_kw2, zorder=100)
    #ax[i,0].text(95,27, ('RMSE = ' + "{:.2f}".format(rmse_ocn[key]['spd_1000'][season].values) + 'm/s'), **text_kw3, zorder=100)
    
    # sst
    cf2=ax[i,1].pcolormesh(lons, lats, obs_diff_mask[key]['sst'][season], cmap=scmap, norm=snorm, transform=trans)
    ax[i,1].text(120,27, ('RMSE = ' + "{:.2f}".format(rmse[key]['sst'][season].values) + '°C'), **text_kw2, zorder=100)
    #ax[i,1].text(95,27, ('RMSE = ' + "{:.2f}".format(rmse_ocn[key]['sst'][season].values) + '°C'), **text_kw3, zorder=100)
    
    # precip
    cf3=ax[i,2].pcolormesh(lons, lats, obs_diff_mask[key]['prec'][season], cmap=pcmap, norm=pnorm, transform=trans)
    ax[i,2].text(120,27, ('RMSE = ' + "{:.2f}".format(rmse[key]['prec'][season].values) + 'mm/day'), **text_kw2, zorder=100)
    #ax[i,2].text(95,27, ('RMSE = ' + "{:.2f}".format(rmse_ocn[key]['prec'][season].values) + 'mm/day'), **text_kw3, zorder=100)
    
    if key in ['ca1senv']:
        for j in [0,1,2]:
            ax[1,j].contour(mlons, mlats, c1env_mask, colors='k', linewidths=3, levels=np.linspace(150,10000,2), transform=trans, zorder=10)
            
    if key in ['ca+sa1senv']:
        for j in [0,1,2]:
            ax[2,j].contour(mlons, mlats, cs1env_mask, colors='k', linewidths=3, levels=np.linspace(150,10000,2), transform=trans, zorder=10)
            
qk=ax[0,0].quiverkey(q1, .8, -0.09, key_length, rf'{key_length} m/s', labelcolor='k', labelpos='E', fontproperties={'size':'large', 'weight':'bold'})

letters = ['a.', 'b.', 'c.', 'd.','e.','f.','g.','h.','i.','j.','k.','l.','m.']
for i, ax in enumerate(ax.flat):
    ax.text(31,27, letters[i], **text_kw,zorder=100)
    # map properties
    ax.coastlines(color='k')
    ax.set_extent(bounds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    if i in [0,3]:
        gl.bottom_labels=False; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    if i in [1,2,4,5]:
        gl.bottom_labels=False; gl.left_labels=False; gl.top_labels=False; gl.right_labels=False
    if i==6:
        gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    if i in [7,8]:
        gl.bottom_labels=True; gl.left_labels=False; gl.top_labels=False; gl.right_labels=False

# add colorbar
cax=fig.add_axes([.07, -0.04, 0.285, 0.02])
cbar=fig.colorbar(cf1, ticks=[-4, -3, -2, -1, 0, 1, 2, 3, 4], orientation='horizontal', extend='both', cax=cax)
cbar.set_label('[m/s]', rotation=0, size=12, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=12)
for tick in cbar.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')
    
cax=fig.add_axes([.385, -0.04, 0.285, 0.02])
cbar=fig.colorbar(cf2, ticks=[-3, -2, -1, 0, 1, 2, 3], orientation='horizontal', extend='both', cax=cax)
cbar.set_label('[°C]', rotation=0, size=12, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=12)
for tick in cbar.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')
    
cax=fig.add_axes([.7, -0.04, 0.285, 0.02])
cbar=fig.colorbar(cf3, ticks=[-6, -4, -2, 0, 2, 4, 6], orientation='horizontal', extend='both', cax=cax)
cbar.set_label('[mm/day]', rotation=0, size=12, fontweight='normal', ha='center')
cbar.ax.tick_params(labelsize=12)
for tick in cbar.ax.xaxis.get_major_ticks():
    tick.label1.set_fontweight('normal')

#plt.savefig(f'aogcm.vsobs.{season}.masked.v3.pdf', transparent=True, bbox_inches='tight')